This notebook has the code for testing three methods of encoding the geohashes, if data is combined for all geohashes for training


1.   Target Encoding (mean encoding)
2.   Leave-One-Out Encoding
3.   Bayesian Target Encoding



In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## Note any model layout/strucutre can be used here with a 1D input array.
def build_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

def train_and_evaluate(X_train_scaled, y_train, X_test_scaled, y_test):
    model = build_model(X_train_scaled.shape[1])
    model.fit(X_train_scaled, y_train, epochs=600, batch_size=128, verbose=0)
    mse = model.evaluate(X_test_scaled, y_test, verbose=0)
    return mse

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import category_encoders as ce

## TRAIN TEST SPLIT
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Separate features and target variable
X_train = train_df.drop(columns=['Time_to_Next_Ride'])
y_train = train_df['Time_to_Next_Ride']
X_test = test_df.drop(columns=['Time_to_Next_Ride'])
y_test = test_df['Time_to_Next_Ride']

In [ ]:
# 1. Target Encoding (Mean Encoding)
mean_encoder = ce.TargetEncoder()
X_train1_encoded = mean_encoder.fit_transform(X_train['Dropoff_Geohash'], y_train)
X_test1_encoded = mean_encoder.transform(X_test['Dropoff_Geohash'])
X_train1 = X_train.copy()
X_test1 = X_test.copy()
X_train1['Dropoff_Geohash_Encoded'] = X_train1_encoded
X_test1['Dropoff_Geohash_Encoded'] = X_test1_encoded
X_train1 = X_train1.drop(columns=['Dropoff_Geohash'])
X_test1 = X_test1.drop(columns=['Dropoff_Geohash'])
scaler1 = StandardScaler()
X_train1_scaled = scaler1.fit_transform(X_train1)
X_test1_scaled = scaler1.transform(X_test1)
mse1 = train_and_evaluate(X_train1_scaled, y_train, X_test1_scaled, y_test)
print(f'Target Encoding Mean Squared Error: {mse1:.4f}')

# 2. Leave-One-Out Encoding
loo_encoder = ce.leave_one_out.LeaveOneOutEncoder(cols=['Dropoff_Geohash'])
X_train2_encoded = loo_encoder.fit_transform(X_train, y_train)
X_test2_encoded = loo_encoder.transform(X_test)
scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2_encoded)
X_test2_scaled = scaler2.transform(X_test2_encoded)
mse2 = train_and_evaluate(X_train2_scaled, y_train, X_test2_scaled, y_test)
print(f'Leave-One-Out Encoding Mean Squared Error: {mse2:.4f}')

# 3. Bayesian Target Encoding
class BayesianTargetEncoder:
    def __init__(self, prior_weight=10):
        self.prior_weight = prior_weight
        self.global_mean = None
        self.category_means = None

    def fit(self, X, y):
        self.global_mean = y.mean()
        category_sums = y.groupby(X).sum()
        category_counts = y.groupby(X).count()
        self.category_means = (category_sums + self.global_mean * self.prior_weight) / (category_counts + self.prior_weight)

    def transform(self, X):
        return X.map(self.category_means).fillna(self.global_mean)

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

bayesian_encoder = BayesianTargetEncoder()
X_train3_encoded = bayesian_encoder.fit_transform(X_train['Dropoff_Geohash'], y_train)
X_test3_encoded = bayesian_encoder.transform(X_test['Dropoff_Geohash'])
X_train3 = X_train.copy()
X_test3 = X_test.copy()
X_train3['Dropoff_Geohash_Encoded'] = X_train3_encoded
X_test3['Dropoff_Geohash_Encoded'] = X_test3_encoded
X_train3 = X_train3.drop(columns=['Dropoff_Geohash'])
X_test3 = X_test3.drop(columns=['Dropoff_Geohash'])
scaler3 = StandardScaler()
X_train3_scaled = scaler3.fit_transform(X_train3)
X_test3_scaled = scaler3.transform(X_test3)
mse3 = train_and_evaluate(X_train3_scaled, y_train, X_test3_scaled, y_test)
print(f'Bayesian Target Encoding Mean Squared Error: {mse3:.4f}')